# Customer Engagement & Product Utilization Analytics


In [1]:
# Cell 1 — Install the packages needed for analysis and the dashboard.
%pip -q install "streamlit>=1.40" "plotly>=5.18" "scipy>=1.10"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 73.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 73.0 MB/s eta 0:00:00


In [2]:
# Cell 2 — Upload the CSV once. The dashboard will reuse the same file.
from pathlib import Path
import shutil
import pandas as pd
import numpy as np

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

WORK_DIR = Path('/content') if IN_COLAB else Path.cwd()
DATA_PATH = WORK_DIR / 'European_Bank.csv'

# Reuse an already-uploaded European_Bank CSV if one exists.
existing = sorted(WORK_DIR.glob('European_Bank*.csv'))
if existing:
    source_path = existing[0]
else:
    if IN_COLAB:
        print('Please upload European_Bank(2).csv (or your bank customer CSV).')
        uploaded = files.upload()
        csv_files = [name for name in uploaded if name.lower().endswith('.csv')]
        if not csv_files:
            raise ValueError('No CSV was uploaded. Run this cell again and select the bank dataset.')
        source_path = WORK_DIR / csv_files[0]
    else:
        csv_files = sorted(WORK_DIR.glob('*.csv'))
        if not csv_files:
            raise FileNotFoundError('Place the bank CSV in this folder and rerun the notebook.')
        source_path = csv_files[0]

if source_path.resolve() != DATA_PATH.resolve():
    shutil.copy2(source_path, DATA_PATH)

df = pd.read_csv(DATA_PATH)
print(f'Loaded: {DATA_PATH.name} | {df.shape[0]:,} rows × {df.shape[1]} columns')
display(df.head())

Loaded: European_Bank.csv | 10,000 rows × 14 columns


,Year,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,2025,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2025,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,2025,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,2025,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,2025,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
# Cell 3 — Validate the schema and show data-quality issues.
from IPython.display import display

REQUIRED_COLUMNS = [
    'Year', 'CustomerId', 'Surname', 'CreditScore', 'Geography', 'Gender', 'Age',
    'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember',
    'EstimatedSalary', 'Exited'
]
NUMERIC_COLUMNS = [
    'Year', 'CustomerId', 'CreditScore', 'Age', 'Tenure', 'Balance',
    'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited'
]
missing_columns = [col for col in REQUIRED_COLUMNS if col not in df.columns]
if missing_columns:
    raise ValueError(f'The CSV is missing required columns: {missing_columns}')

analysis_df = df.copy()
for col in NUMERIC_COLUMNS:
    analysis_df[col] = pd.to_numeric(analysis_df[col], errors='coerce')

binary_checks = {
    'Exited': {0, 1},
    'HasCrCard': {0, 1},
    'IsActiveMember': {0, 1},
}
quality_rows = [
    ('Rows', len(df)),
    ('Columns', df.shape[1]),
    ('Missing cells', int(df.isna().sum().sum())),
    ('Exact duplicate rows', int(df.duplicated().sum())),
    ('Duplicate CustomerId values', int(df['CustomerId'].duplicated().sum())),
    ('Rows with missing or non-numeric required numeric values', int(analysis_df[NUMERIC_COLUMNS].isna().any(axis=1).sum())),
    ('Invalid ages (outside 18–100)', int((~analysis_df['Age'].between(18, 100)).fillna(True).sum())),
    ('Negative balances', int((analysis_df['Balance'] < 0).fillna(False).sum())),
    ('Negative estimated salaries', int((analysis_df['EstimatedSalary'] < 0).fillna(False).sum())),
    ('Unsupported product counts (expected 1–4)', int((~analysis_df['NumOfProducts'].isin([1, 2, 3, 4])).fillna(True).sum())),
]
for col, allowed in binary_checks.items():
    quality_rows.append((f'Invalid {col} values', int((~analysis_df[col].isin(allowed)).sum())))
quality_table = pd.DataFrame(quality_rows, columns=['Check', 'Result'])

data = analysis_df[
    analysis_df['Exited'].isin([0, 1])
    & analysis_df['IsActiveMember'].isin([0, 1])
    & analysis_df['NumOfProducts'].isin([1, 2, 3, 4])
    & analysis_df['HasCrCard'].isin([0, 1])
].copy()

print('Data-quality checks (issues are reported, not silently corrected):')
display(quality_table)
print(f'Rows used for the main churn analysis: {len(data):,} of {len(df):,}')
if data.empty:
    raise ValueError('No rows have valid churn, activity, product-count, and card-ownership values.')
if data['Year'].nunique(dropna=True) == 1:
    print(f"Note: every valid row has Year = {data['Year'].dropna().iloc[0]:g}; year-over-year analysis is not appropriate.")

Data-quality checks (issues are reported, not silently corrected):


,Check,Result
0,Rows,10000
1,Columns,14
2,Missing cells,0
3,Exact duplicate rows,0
4,Duplicate CustomerId values,0
5,Rows with missing or non-numeric required nume...,0
6,Invalid ages (outside 18–100),0
7,Negative balances,0
8,Negative estimated salaries,0
9,Unsupported product counts (expected 1–4),0


Rows used for the main churn analysis: 10,000 of 10,000
Note: every valid row has Year = 2025; year-over-year analysis is not appropriate.


In [4]:
# Cell 4 — Main KPIs and segment tables.
from math import sqrt

n_customers = int(len(data))
n_churned = int(data['Exited'].sum())
n_retained = int(n_customers - n_churned)
churn_rate = n_churned / n_customers if n_customers else np.nan
retention_rate = 1 - churn_rate if n_customers else np.nan

# Wilson 95% confidence interval for a binomial proportion.
def wilson_ci(successes, total, z=1.96):
    if total == 0:
        return (np.nan, np.nan)
    p = successes / total
    den = 1 + z*z/total
    center = (p + z*z/(2*total)) / den
    half = z * sqrt((p*(1-p) + z*z/(4*total))/total) / den
    return (max(0, center-half), min(1, center+half))

def rate_table(frame, column, order=None):
    result = (frame.groupby(column, dropna=False, observed=True)['Exited']
              .agg(customers='size', churned='sum', churn_rate='mean')
              .reset_index())
    result['retained'] = result['customers'] - result['churned']
    result['retention_rate'] = 1 - result['churn_rate']
    ci = result.apply(lambda row: wilson_ci(int(row['churned']), int(row['customers'])), axis=1)
    result['churn_ci_low'] = [x[0] for x in ci]
    result['churn_ci_high'] = [x[1] for x in ci]
    if order is not None:
        result[column] = pd.Categorical(result[column], categories=order, ordered=True)
        result = result.sort_values(column).reset_index(drop=True)
    return result

overall_kpis = pd.DataFrame({
    'Metric': ['Customers analyzed', 'Retained customers', 'Churned customers', 'Churn rate', 'Retention rate'],
    'Value': [n_customers, n_retained, n_churned, churn_rate, retention_rate]
})
activity_table = rate_table(data, 'IsActiveMember', [0, 1])
activity_table['Activity'] = activity_table['IsActiveMember'].map({0: 'Inactive', 1: 'Active'})
product_table = rate_table(data, 'NumOfProducts', [1, 2, 3, 4])
card_table = rate_table(data, 'HasCrCard', [0, 1])
card_table['Card ownership'] = card_table['HasCrCard'].map({0: 'No card', 1: 'Cardholder'})

balance_threshold = float(data['Balance'].quantile(0.80))
high_balance = data[data['Balance'] >= balance_threshold].copy()
inactive_high_balance = high_balance[high_balance['IsActiveMember'] == 0].copy()
high_balance_disengagement = len(inactive_high_balance) / len(high_balance) if len(high_balance) else np.nan
inactive_active_ratio = (
    float(activity_table.loc[activity_table['IsActiveMember'] == 0, 'churn_rate'].iloc[0]) /
    float(activity_table.loc[activity_table['IsActiveMember'] == 1, 'churn_rate'].iloc[0])
    if (activity_table['IsActiveMember'] == 0).any() and
       (activity_table.loc[activity_table['IsActiveMember'] == 1, 'churn_rate'].iloc[0] != 0)
    else np.nan
)
retention_card = float(card_table.loc[card_table['HasCrCard'] == 1, 'retention_rate'].iloc[0])
retention_no_card = float(card_table.loc[card_table['HasCrCard'] == 0, 'retention_rate'].iloc[0])
card_stickiness_pp = (retention_card - retention_no_card) * 100

print('Overall KPIs')
display(overall_kpis)
print('Churn by activity status (rates include 95% confidence intervals)')
display(activity_table[['Activity', 'customers', 'retained', 'churned', 'churn_rate', 'churn_ci_low', 'churn_ci_high']])
print('Churn by individual product count')
display(product_table[['NumOfProducts', 'customers', 'retained', 'churned', 'churn_rate', 'churn_ci_low', 'churn_ci_high']])
print('Churn by card ownership')
display(card_table[['Card ownership', 'customers', 'retained', 'churned', 'churn_rate', 'retention_rate']])
print(f'80th-percentile balance threshold: {balance_threshold:,.2f} (currency unspecified)')
print(f'High-balance disengagement rate: {high_balance_disengagement:.2%}; inactive high-balance customers: {len(inactive_high_balance):,}')
print(f'Inactive-to-active churn-rate ratio: {inactive_active_ratio:.2f}x')
print(f'Cardholder minus non-cardholder retention difference: {card_stickiness_pp:+.2f} percentage points')

Overall KPIs


,Metric,Value
0,Customers analyzed,10000.0000
1,Retained customers,7963.0000
2,Churned customers,2037.0000
3,Churn rate,0.2037
4,Retention rate,0.7963


Churn by activity status (rates include 95% confidence intervals)


,Activity,customers,retained,churned,churn_rate,churn_ci_low,churn_ci_high
0,Inactive,4849,3547,1302,0.268509,0.256222,0.281163
1,Active,5151,4416,735,0.142691,0.133405,0.152509


Churn by individual product count


,NumOfProducts,customers,retained,churned,churn_rate,churn_ci_low,churn_ci_high
0,1,5084,3675,1409,0.277144,0.265012,0.289612
1,2,4590,4242,348,0.075817,0.068509,0.083835
2,3,266,46,220,0.827068,0.777047,0.867775
3,4,60,0,60,1.000000,0.939826,1.000000


Churn by card ownership


,Card ownership,customers,retained,churned,churn_rate,retention_rate
0,No card,2945,2332,613,0.208149,0.791851
1,Cardholder,7055,5631,1424,0.201843,0.798157


80th-percentile balance threshold: 133,710.36 (currency unspecified)
High-balance disengagement rate: 49.25%; inactive high-balance customers: 985
Inactive-to-active churn-rate ratio: 1.88x
Cardholder minus non-cardholder retention difference: +0.63 percentage points


In [5]:
# Cell 5 — Statistical checks and supporting variables.
from scipy.stats import chi2_contingency, fisher_exact

def association_test(frame, column):
    table = pd.crosstab(frame[column], frame['Exited'])
    if table.shape[0] < 2 or table.shape[1] < 2:
        return {'Variable': column, 'Test': 'Not applicable', 'Statistic': np.nan, 'p_value': np.nan, 'Note': 'Only one observed category'}
    chi2, p_value, dof, expected = chi2_contingency(table)
    test_name = 'Chi-square'
    stat = chi2
    note = 'Association only; not causation.'
    if table.shape == (2, 2):
        stat, p_value = fisher_exact(table.to_numpy())
        test_name = "Fisher's exact"
        note = 'Exact 2×2 association test; not causation.'
    if (expected < 5).any():
        note += ' Some expected cell counts are below 5; interpret carefully.'
    return {'Variable': column, 'Test': test_name, 'Statistic': stat, 'p_value': p_value, 'Note': note}

stat_results = pd.DataFrame([
    association_test(data, 'IsActiveMember'),
    association_test(data, 'NumOfProducts'),
    association_test(data, 'HasCrCard'),
    association_test(data, 'Geography'),
])
stat_results['p_value'] = stat_results['p_value'].map(lambda p: f'{p:.4g}' if pd.notna(p) else 'N/A')
print('Statistical association checks')
display(stat_results)

support_df = data.copy()
support_df['AgeBand'] = pd.cut(support_df['Age'], bins=[17, 29, 39, 49, 59, 100], labels=['18–29', '30–39', '40–49', '50–59', '60+'])
support_df['TenureBand'] = pd.cut(support_df['Tenure'], bins=[-1, 2, 5, 8, 10, 100], labels=['0–2', '3–5', '6–8', '9–10', '11+'])
support_df['CreditScoreBand'] = pd.cut(support_df['CreditScore'], bins=[-1, 499, 599, 699, 799, 10000], labels=['<500', '500–599', '600–699', '700–799', '800+'])
age_table = rate_table(support_df, 'AgeBand')
tenure_table = rate_table(support_df, 'TenureBand')
geo_table = rate_table(support_df, 'Geography')
credit_table = rate_table(support_df, 'CreditScoreBand')
print('Supporting comparisons (context only; primary focus remains activity and product relationships)')
display(geo_table[['Geography', 'customers', 'churn_rate', 'retention_rate']])
display(age_table[['AgeBand', 'customers', 'churn_rate', 'retention_rate']])
display(tenure_table[['TenureBand', 'customers', 'churn_rate', 'retention_rate']])
display(credit_table[['CreditScoreBand', 'customers', 'churn_rate', 'retention_rate']])

Statistical association checks


,Variable,Test,Statistic,p_value,Note
0,IsActiveMember,Fisher's exact,0.453428,2.294e-55,Exact 2×2 association test; not causation.
1,NumOfProducts,Chi-square,1503.629362,0,Association only; not causation.
2,HasCrCard,Fisher's exact,0.962039,0.4789,Exact 2×2 association test; not causation.
3,Geography,Chi-square,301.255337,3.83e-66,Association only; not causation.


Supporting comparisons (context only; primary focus remains activity and product relationships)


,Geography,customers,churn_rate,retention_rate
0,France,5014,0.161548,0.838452
1,Germany,2509,0.324432,0.675568
2,Spain,2477,0.166734,0.833266


,AgeBand,customers,churn_rate,retention_rate
0,18–29,1641,0.075564,0.924436
1,30–39,4346,0.108836,0.891164
2,40–49,2618,0.307869,0.692131
3,50–59,869,0.560414,0.439586
4,60+,526,0.279468,0.720532


,TenureBand,customers,churn_rate,retention_rate
0,0–2,2496,0.211538,0.788462
1,3–5,3010,0.207641,0.792359
2,6–8,3020,0.188742,0.811258
3,9–10,1474,0.213026,0.786974


,CreditScoreBand,customers,churn_rate,retention_rate
0,<500,632,0.237342,0.762658
1,500–599,2402,0.212323,0.787677
2,600–699,3818,0.197224,0.802776
3,700–799,2493,0.198957,0.801043
4,800+,655,0.195420,0.804580


In [6]:
# Cell 6 — Create and display the analysis graphs.
import plotly.express as px
from IPython.display import display

# 1. Activity and churn
activity_chart = px.bar(
    activity_table, x='Activity', y='churn_rate', color='Activity',
    hover_data=['customers', 'churn_ci_low', 'churn_ci_high'],
    title='1. Observed churn by customer activity',
    labels={'churn_rate': 'Churn rate', 'Activity': 'Activity status', 'customers': 'Customers'}
)
activity_chart.update_yaxes(tickformat='.0%', rangemode='tozero')

# 2. Individual product-count groups (keep 3 and 4 products separate)
product_chart = px.bar(
    product_table, x='NumOfProducts', y='churn_rate', text='customers',
    hover_data=['retention_rate', 'churn_ci_low', 'churn_ci_high'],
    title='2. Observed churn by individual product count',
    labels={'NumOfProducts': 'Number of products', 'churn_rate': 'Churn rate', 'customers': 'Customers'}
)
product_chart.update_yaxes(tickformat='.0%', rangemode='tozero')

# 3. Credit-card ownership
card_chart = px.bar(
    card_table, x='Card ownership', y='churn_rate', text='customers',
    hover_data=['retention_rate'], title='3. Observed churn by credit-card ownership',
    labels={'churn_rate': 'Churn rate'}
)
card_chart.update_yaxes(tickformat='.0%', rangemode='tozero')

# 4. Balance by activity and churn outcome
balance_chart = px.box(
    data, x='IsActiveMember', y='Balance', color='Exited', points=False,
    title='4. Balance distribution by activity and churn',
    labels={'IsActiveMember': 'Activity (0 = inactive, 1 = active)',
            'Balance': 'Balance (currency unspecified)', 'Exited': 'Churned'}
)

# 5. Geography
geography_chart = px.bar(
    geo_table, x='Geography', y='churn_rate', text='customers',
    hover_data=['retention_rate'], title='5. Observed churn by geography',
    labels={'churn_rate': 'Churn rate', 'customers': 'Customers'}
)
geography_chart.update_yaxes(tickformat='.0%', rangemode='tozero')

# 6. Age bands
age_chart = px.bar(
    age_table, x='AgeBand', y='churn_rate', text='customers',
    hover_data=['retention_rate'], title='6. Observed churn by age band',
    labels={'AgeBand': 'Age band', 'churn_rate': 'Churn rate', 'customers': 'Customers'}
)
age_chart.update_yaxes(tickformat='.0%', rangemode='tozero')

# 7. Tenure bands
tenure_chart = px.bar(
    tenure_table, x='TenureBand', y='churn_rate', text='customers',
    hover_data=['retention_rate'], title='7. Observed churn by tenure band',
    labels={'TenureBand': 'Tenure with bank (years)', 'churn_rate': 'Churn rate', 'customers': 'Customers'}
)
tenure_chart.update_yaxes(tickformat='.0%', rangemode='tozero')

# 8. Credit-score bands
credit_score_chart = px.bar(
    credit_table, x='CreditScoreBand', y='churn_rate', text='customers',
    hover_data=['retention_rate'], title='8. Observed churn by credit-score band',
    labels={'CreditScoreBand': 'Credit-score band', 'churn_rate': 'Churn rate', 'customers': 'Customers'}
)
credit_score_chart.update_yaxes(tickformat='.0%', rangemode='tozero')

# 9. Customer counts by product count and churn outcome
product_distribution_chart = px.histogram(
    data, x='NumOfProducts', color='Exited', barmode='group',
    category_orders={'NumOfProducts': [1, 2, 3, 4], 'Exited': [0, 1]},
    title='9. Customer count by product count and churn outcome',
    labels={'NumOfProducts': 'Number of products', 'count': 'Customers', 'Exited': 'Churned (0 = no, 1 = yes)'}
)

# 10. Salary versus balance; sample only for a readable interactive figure
scatter_data = data.sample(n=min(len(data), 3000), random_state=42) if len(data) else data.copy()
salary_balance_chart = px.scatter(
    scatter_data, x='EstimatedSalary', y='Balance', color='Exited',
    symbol='IsActiveMember', opacity=0.65,
    hover_data=['NumOfProducts', 'Geography'],
    title='10. Salary versus balance by churn and activity (sample up to 3,000 customers)',
    labels={'EstimatedSalary': 'Estimated salary (currency unspecified)',
            'Balance': 'Balance (currency unspecified)', 'Exited': 'Churned',
            'IsActiveMember': 'Active member'}
)

figure_objects = {
    'churn_by_activity': activity_chart,
    'churn_by_product_count': product_chart,
    'churn_by_card_ownership': card_chart,
    'balance_activity_churn': balance_chart,
    'churn_by_geography': geography_chart,
    'churn_by_age_band': age_chart,
    'churn_by_tenure_band': tenure_chart,
    'churn_by_credit_score_band': credit_score_chart,
    'customer_count_by_product_and_churn': product_distribution_chart,
    'salary_vs_balance': salary_balance_chart,
}

for chart in figure_objects.values():
    chart.show()

In [7]:
# Cell 7 — Transparent relationship score and a customer-review shortlist.
SCORE_WEIGHTS = {'activity': 50, 'product_breadth': 35, 'card_ownership': 15}
scored_data = data.copy()
scored_data['activity_component'] = scored_data['IsActiveMember'].astype(float)
scored_data['product_component'] = scored_data['NumOfProducts'].clip(upper=2).astype(float) / 2
scored_data['card_component'] = scored_data['HasCrCard'].astype(float)
scored_data['RelationshipStrengthIndex'] = (
    SCORE_WEIGHTS['activity'] * scored_data['activity_component']
    + SCORE_WEIGHTS['product_breadth'] * scored_data['product_component']
    + SCORE_WEIGHTS['card_ownership'] * scored_data['card_component']
)
scored_data['ScoreBand'] = pd.cut(
    scored_data['RelationshipStrengthIndex'], bins=[-0.01, 25, 50, 75, 100.01],
    labels=['0–25', '26–50', '51–75', '76–100'], include_lowest=True
)

def priority_label(row):
    if row['IsActiveMember'] == 0 and row['Balance'] >= balance_threshold:
        return 'Priority 1 — inactive, high balance'
    if row['NumOfProducts'] >= 3:
        return 'Review — 3 or 4 products'
    if row['IsActiveMember'] == 0:
        return 'Review — inactive customer'
    if row['NumOfProducts'] == 1:
        return 'Opportunity — active, one product'
    return 'Standard monitoring'

scored_data['RetentionPriority'] = scored_data.apply(priority_label, axis=1)
score_summary = (scored_data.groupby('ScoreBand', observed=False)
                 .agg(customers=('CustomerId', 'size'), churn_rate=('Exited', 'mean'),
                      retention_rate=('Exited', lambda s: 1-s.mean()), average_balance=('Balance', 'mean'))
                 .reset_index())
priority_summary = (scored_data.groupby('RetentionPriority', dropna=False)
                    .agg(customers=('CustomerId', 'size'), churn_rate=('Exited', 'mean'),
                         retention_rate=('Exited', lambda s: 1-s.mean()), average_balance=('Balance', 'mean'))
                    .reset_index().sort_values('customers', ascending=False))

# No surnames in the follow-up table. Keep only fields useful for review.
customer_shortlist = scored_data[scored_data['RetentionPriority'] != 'Standard monitoring'][[
    'CustomerId', 'Geography', 'Age', 'Balance', 'EstimatedSalary', 'NumOfProducts',
    'IsActiveMember', 'HasCrCard', 'Exited', 'RelationshipStrengthIndex', 'RetentionPriority'
]].sort_values(['RetentionPriority', 'Balance'], ascending=[True, False])

print('Relationship Strength Index (0–100; heuristic, not churn probability)')
display(score_summary)
print('Retention review groups (priority labels use a documented precedence order)')
display(priority_summary)
print(f'Customer records flagged for review: {len(customer_shortlist):,}')
display(customer_shortlist.head(10))

# Two additional charts based on the relationship score and retention-review groups.
score_band_chart = px.bar(
    score_summary, x='ScoreBand', y='churn_rate', text='customers',
    hover_data=['retention_rate', 'average_balance'],
    title='11. Observed churn by Relationship Strength Index band',
    labels={'ScoreBand': 'Relationship Strength Index band', 'churn_rate': 'Churn rate', 'customers': 'Customers'}
)
score_band_chart.update_yaxes(tickformat='.0%', rangemode='tozero')
score_band_chart.show()

priority_chart = px.bar(
    priority_summary, x='RetentionPriority', y='churn_rate', text='customers',
    hover_data=['retention_rate', 'average_balance'],
    title='12. Observed churn by retention-review group',
    labels={'RetentionPriority': 'Retention-review group', 'churn_rate': 'Churn rate', 'customers': 'Customers'}
)
priority_chart.update_yaxes(tickformat='.0%', rangemode='tozero')
priority_chart.update_xaxes(tickangle=-20)
priority_chart.show()


Relationship Strength Index (0–100; heuristic, not churn probability)


,ScoreBand,customers,churn_rate,retention_rate,average_balance
0,0–25,718,0.345404,0.654596,100598.780432
1,26–50,4131,0.255144,0.744856,73056.080625
2,51–75,788,0.215736,0.784264,100577.892602
3,76–100,4363,0.129498,0.870502,71413.917052


Retention review groups (priority labels use a documented precedence order)


,RetentionPriority,customers,churn_rate,retention_rate,average_balance
3,Review — inactive customer,3718,0.236148,0.763852,57333.207534
0,"Opportunity — active, one product",2563,0.189231,0.810769,98902.019317
4,Standard monitoring,2446,0.055601,0.944399,51852.938729
1,"Priority 1 — inactive, high balance",985,0.300508,0.699492,153745.862223
2,Review — 3 or 4 products,288,0.840278,0.159722,69222.879653


Customer records flagged for review: 7,554


,CustomerId,Geography,Age,Balance,EstimatedSalary,NumOfProducts,IsActiveMember,HasCrCard,Exited,RelationshipStrengthIndex,RetentionPriority
3280,15715622,France,57,238387.56,147964.99,1,1,0,1,67.5,"Opportunity — active, one product"
6717,15586674,Spain,58,216109.88,74176.71,1,1,0,1,67.5,"Opportunity — active, one product"
1533,15769818,France,37,212778.20,69372.88,1,1,0,0,67.5,"Opportunity — active, one product"
8702,15690589,France,37,212314.03,148814.54,1,1,0,0,67.5,"Opportunity — active, one product"
8794,15578671,Spain,29,209490.21,133267.69,1,1,1,1,82.5,"Opportunity — active, one product"
6913,15784180,France,36,206329.65,46632.87,1,1,1,1,82.5,"Opportunity — active, one product"
9920,15673020,France,49,204510.94,738.88,1,1,0,1,67.5,"Opportunity — active, one product"
6271,15620756,France,49,202904.64,17298.72,1,1,1,1,82.5,"Opportunity — active, one product"
7492,15776545,France,28,200724.96,82872.64,1,1,0,1,67.5,"Opportunity — active, one product"
2597,15668818,Spain,40,200322.45,113244.73,1,1,1,0,82.5,"Opportunity — active, one product"


In [8]:
# Cell 8 — Save charts and aggregate analysis tables for the outputs ZIP.
from pathlib import Path

OUTPUT_DIR = WORK_DIR / 'retention_analytics_outputs'
FIGURE_DIR = OUTPUT_DIR / 'figures'
TABLE_DIR = OUTPUT_DIR / 'tables'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

# Remove stale CSV/HTML outputs from earlier runs so the ZIP contains only current results.
for old_file in list(TABLE_DIR.glob('*.csv')) + list(FIGURE_DIR.glob('*.html')):
    old_file.unlink()

# Remove any stale research-paper files left by an earlier notebook version.
for stale_file in list(OUTPUT_DIR.glob('*.docx')) + list(OUTPUT_DIR.glob('*.md')):
    stale_file.unlink()

# Aggregate tables only. The raw customer CSV and customer-level shortlist are not included.
output_tables = {
    'overall_kpis.csv': overall_kpis,
    'data_quality.csv': quality_table,
    'churn_by_activity.csv': activity_table,
    'churn_by_product_count.csv': product_table,
    'churn_by_card_ownership.csv': card_table,
    'statistical_tests.csv': stat_results,
    'retention_priority_summary.csv': priority_summary,
    'relationship_score_summary.csv': score_summary,
    'geography_summary.csv': geo_table,
    'age_summary.csv': age_table,
    'tenure_summary.csv': tenure_table,
    'credit_score_summary.csv': credit_table,
}
for filename, table in output_tables.items():
    table.to_csv(TABLE_DIR / filename, index=False)

# Save each chart as an interactive HTML file so it can be reopened in a browser.
figure_objects['relationship_strength_by_churn'] = score_band_chart
figure_objects['retention_priority_by_churn'] = priority_chart
for name, figure in figure_objects.items():
    figure.write_html(FIGURE_DIR / f'{name}.html', include_plotlyjs=True, full_html=True)

print(f'Saved {len(output_tables)} aggregate tables to: {TABLE_DIR}')
print(f'Saved {len(figure_objects)} interactive graphs to: {FIGURE_DIR}')
print('Note: the outputs contain aggregate results and charts, not the raw customer CSV or customer-level records.')

Saved 12 aggregate tables to: /content/retention_analytics_outputs/tables
Saved 12 interactive graphs to: /content/retention_analytics_outputs/figures
Note: the outputs contain aggregate results and charts, not the raw customer CSV or customer-level records.


In [9]:
%%writefile /content/retention_dashboard.py
from pathlib import Path
import io
import numpy as np
import pandas as pd
import plotly.express as px
import streamlit as st

st.set_page_config(page_title='Customer Retention Analytics', page_icon='📊', layout='wide')
st.title('Customer Engagement & Product Utilization Analytics')
st.caption('Observational retention analysis • results are associations, not proof of causation')

DATA_PATH = Path('/content/European_Bank.csv')
uploaded = st.sidebar.file_uploader('Upload customer CSV (optional if already uploaded in Colab)', type=['csv'])
try:
    if uploaded is not None:
        df = pd.read_csv(uploaded)
    elif DATA_PATH.exists():
        df = pd.read_csv(DATA_PATH)
    else:
        st.info('Upload the bank customer CSV in the sidebar to begin.')
        st.stop()
except Exception as exc:
    st.error(f'Could not read the CSV: {exc}')
    st.stop()

required = ['CustomerId', 'Geography', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited']
missing = [c for c in required if c not in df.columns]
if missing:
    st.error(f'Missing required columns: {missing}')
    st.stop()

for col in ['Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited', 'CustomerId', 'CreditScore', 'Year']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
data = df[df['Exited'].isin([0, 1]) & df['IsActiveMember'].isin([0, 1]) & df['HasCrCard'].isin([0, 1]) & df['NumOfProducts'].isin([1, 2, 3, 4])].copy()
if data.empty:
    st.error('No rows have valid churn, activity, card, and product-count values.')
    st.stop()

st.sidebar.header('Filters')
geographies = sorted(data['Geography'].dropna().astype(str).unique().tolist())
selected_geo = st.sidebar.multiselect('Geography', geographies, default=geographies)
activity_choice = st.sidebar.selectbox('Activity status', ['All', 'Active', 'Inactive'])
product_choices = sorted(data['NumOfProducts'].dropna().astype(int).unique().tolist())
selected_products = st.sidebar.multiselect('Number of products', product_choices, default=product_choices)
card_choice = st.sidebar.selectbox('Credit-card ownership', ['All', 'Has card', 'No card'])
filtered = data[data['Geography'].astype(str).isin(selected_geo) & data['NumOfProducts'].isin(selected_products)].copy()
if activity_choice == 'Active':
    filtered = filtered[filtered['IsActiveMember'] == 1]
elif activity_choice == 'Inactive':
    filtered = filtered[filtered['IsActiveMember'] == 0]
if card_choice == 'Has card':
    filtered = filtered[filtered['HasCrCard'] == 1]
elif card_choice == 'No card':
    filtered = filtered[filtered['HasCrCard'] == 0]
if filtered.empty:
    st.warning('No customers match these filters. Change the filters in the sidebar.')
    st.stop()

churn = float(filtered['Exited'].mean())
retention = 1 - churn
active = int((filtered['IsActiveMember'] == 1).sum())
inactive = int((filtered['IsActiveMember'] == 0).sum())
high_balance_cut = float(data['Balance'].quantile(0.80))

m1, m2, m3, m4 = st.columns(4)
m1.metric('Customers in view', f'{len(filtered):,}')
m2.metric('Observed churn rate', f'{churn:.2%}')
m3.metric('Retention rate', f'{retention:.2%}')
m4.metric('Inactive customers', f'{inactive:,}')

# All dashboard sections live in this single Streamlit file.
tabs = st.tabs(['Overview', 'Engagement', 'Product Count', 'Financial Review', 'Relationship Score', 'Data Quality'])

def grouped_rates(frame, col):
    out = frame.groupby(col, dropna=False)['Exited'].agg(customers='size', churn_rate='mean').reset_index()
    out['retained'] = out['customers'] * (1 - out['churn_rate'])
    out['retention_rate'] = 1 - out['churn_rate']
    return out

with tabs[0]:
    st.subheader('Churn overview')
    st.write(f'Current filtered population: **{len(filtered):,}** of **{len(data):,}** valid customer records.')
    c1, c2 = st.columns(2)
    by_activity = grouped_rates(filtered, 'IsActiveMember')
    by_activity['Activity'] = by_activity['IsActiveMember'].map({0: 'Inactive', 1: 'Active'})
    fig = px.bar(by_activity, x='Activity', y='churn_rate', text='customers', title='Churn by activity', labels={'churn_rate':'Churn rate'})
    fig.update_yaxes(tickformat='.0%', rangemode='tozero')
    c1.plotly_chart(fig, use_container_width=True)
    by_product = grouped_rates(filtered, 'NumOfProducts')
    fig2 = px.bar(by_product, x='NumOfProducts', y='churn_rate', text='customers', title='Churn by product count', labels={'NumOfProducts':'Number of products','churn_rate':'Churn rate'})
    fig2.update_yaxes(tickformat='.0%', rangemode='tozero')
    c2.plotly_chart(fig2, use_container_width=True)
    st.caption('High churn in a group is a signal for investigation, not proof that a particular intervention will work.')

with tabs[1]:
    st.subheader('Engagement analytics')
    engagement = grouped_rates(filtered, 'IsActiveMember')
    engagement['Activity'] = engagement['IsActiveMember'].map({0:'Inactive', 1:'Active'})
    st.dataframe(engagement[['Activity','customers','churn_rate','retention_rate']].style.format({'churn_rate':'{:.2%}','retention_rate':'{:.2%}'}), use_container_width=True)
    rates = dict(zip(engagement['IsActiveMember'], engagement['churn_rate']))
    if 0 in rates and 1 in rates:
        st.write(f'Inactive minus active churn difference: **{(rates[0]-rates[1])*100:+.2f} percentage points**.')
        st.write(f'Inactive-to-active churn-rate ratio: **{(rates[0]/rates[1]):.2f}×**.' if rates[1] else 'Active customer churn rate is zero; ratio is undefined.')
    st.info('IsActiveMember is a coarse activity flag. Transaction recency, frequency, product-level use, and channel engagement are not available.')

with tabs[2]:
    st.subheader('Product-count analysis')
    products = grouped_rates(filtered, 'NumOfProducts').sort_values('NumOfProducts')
    st.dataframe(products[['NumOfProducts','customers','churn_rate','retention_rate']].style.format({'churn_rate':'{:.2%}','retention_rate':'{:.2%}'}), use_container_width=True)
    st.warning('Three- and four-product customers should be investigated separately because their group sizes may be much smaller. Product count is not actual utilization.')

with tabs[3]:
    st.subheader('Financial commitment versus engagement')
    percentile = st.slider('High-balance percentile threshold', min_value=50, max_value=95, value=80, step=5)
    threshold = float(data['Balance'].quantile(percentile/100))
    segment = filtered[(filtered['Balance'] >= threshold) & (filtered['IsActiveMember'] == 0)].copy()
    col1, col2, col3 = st.columns(3)
    col1.metric('Balance cutoff', f'{threshold:,.2f}')
    col2.metric('Inactive high-balance customers', f'{len(segment):,}')
    col3.metric('Segment churn rate', f'{segment["Exited"].mean():.2%}' if len(segment) else 'N/A')
    st.caption('Currency is unspecified. Balance is not profitability or lifetime value. This is a review segment, not an indication of customer intent.')
    show_cols = [c for c in ['CustomerId','Geography','Age','Balance','EstimatedSalary','NumOfProducts','IsActiveMember','HasCrCard','Exited'] if c in segment.columns]
    st.dataframe(segment[show_cols].head(500), use_container_width=True)
    st.download_button('Download filtered high-balance customers (CSV)', segment[show_cols].to_csv(index=False).encode('utf-8'), 'inactive_high_balance_customers.csv', 'text/csv')

with tabs[4]:
    st.subheader('Relationship Strength Index')
    st.caption('Heuristic score out of 100, not a validated prediction of churn.')
    w_activity = st.slider('Activity weight', 0, 100, 50)
    w_product = st.slider('Product breadth weight', 0, 100, 35)
    w_card = st.slider('Card ownership weight', 0, 100, 15)
    total_w = w_activity + w_product + w_card
    if total_w == 0:
        st.warning('At least one weight must be above zero.')
    else:
        score_df = filtered.copy()
        product_component = score_df['NumOfProducts'].clip(upper=2) / 2
        score_df['RelationshipStrengthIndex'] = 100 * (
            (w_activity/total_w) * score_df['IsActiveMember']
            + (w_product/total_w) * product_component
            + (w_card/total_w) * score_df['HasCrCard']
        )
        score_df['ScoreBand'] = pd.cut(score_df['RelationshipStrengthIndex'], [-0.01,25,50,75,100.01], labels=['0–25','26–50','51–75','76–100'], include_lowest=True)
        summary = score_df.groupby('ScoreBand', observed=False).agg(customers=('CustomerId','size'), churn_rate=('Exited','mean'), retention_rate=('Exited',lambda x:1-x.mean())).reset_index()
        st.dataframe(summary.style.format({'churn_rate':'{:.2%}','retention_rate':'{:.2%}'}), use_container_width=True)
        fig3 = px.bar(summary, x='ScoreBand', y='churn_rate', text='customers', title='Observed churn by score band', labels={'churn_rate':'Observed churn rate'})
        fig3.update_yaxes(tickformat='.0%', rangemode='tozero')
        st.plotly_chart(fig3, use_container_width=True)
        st.write('Compare score bands with actual observed churn; do not use the score alone to determine retention priority.')

with tabs[5]:
    st.subheader('Data quality & definitions')
    qrows = [
        ('Rows', len(df)), ('Columns', df.shape[1]), ('Missing cells', int(df.isna().sum().sum())),
        ('Exact duplicate rows', int(df.duplicated().sum())), ('Duplicate CustomerId values', int(df['CustomerId'].duplicated().sum())),
        ('Valid analysis rows', len(data)), ('Invalid / excluded rows', len(df)-len(data)),
    ]
    st.dataframe(pd.DataFrame(qrows, columns=['Check','Result']), use_container_width=True)
    st.markdown('**Definitions**')
    st.markdown('- Churn rate = customers with `Exited = 1` / customers in segment.\n- Retention rate = 1 − churn rate.\n- Product breadth score component = `min(NumOfProducts, 2) / 2`.\n- `CustomerId` is an identifier, not an analytical predictor. Surnames are not shown in exports.')
    st.markdown('**Limitations**')
    st.markdown('- The data is observational; results do not prove causality.\n- Credit-card ownership does not indicate usage.\n- Balance and estimated salary are not profitability measures.\n- A single Year value cannot support year-over-year trends.')
    export_cols = [c for c in ['CustomerId','Geography','Age','Tenure','Balance','EstimatedSalary','NumOfProducts','IsActiveMember','HasCrCard','Exited'] if c in filtered.columns]
    st.download_button('Download current filtered records (CSV)', filtered[export_cols].to_csv(index=False).encode('utf-8'), 'filtered_customer_records.csv', 'text/csv')

st.divider()
st.caption('Suggested next step: test outreach or product-fit changes with a controlled comparison group and monitor subsequent behavior.')


Writing /content/retention_dashboard.py


In [10]:
# Cell 10 — Launch Streamlit and show the real Colab browser link (not localhost).
import subprocess, sys, time, urllib.request
from pathlib import Path

PORT = 8501
APP_PATH = Path('/content/retention_dashboard.py') if IN_COLAB else (Path.cwd() / 'retention_dashboard.py')
LOG_PATH = WORK_DIR / 'streamlit_dashboard.log'

# Stop the previous server started by this notebook, if it is still running.
try:
    if 'streamlit_process' in globals() and streamlit_process.poll() is None:
        streamlit_process.terminate()
        streamlit_process.wait(timeout=5)
except Exception:
    try:
        streamlit_process.kill()
    except Exception:
        pass

command = [
    sys.executable, '-m', 'streamlit', 'run', str(APP_PATH),
    '--server.port', str(PORT),
    '--server.address', '0.0.0.0',
    '--server.headless', 'true',
    '--server.enableCORS', 'false',
    '--server.enableXsrfProtection', 'false',
    '--browser.gatherUsageStats', 'false',
]
with open(LOG_PATH, 'w', encoding='utf-8') as log_handle:
    streamlit_process = subprocess.Popen(command, stdout=log_handle, stderr=subprocess.STDOUT)

# Wait until the app health endpoint responds rather than guessing that startup succeeded.
ready = False
for _ in range(30):
    if streamlit_process.poll() is not None:
        break
    try:
        with urllib.request.urlopen(f'http://127.0.0.1:{PORT}/_stcore/health', timeout=1) as response:
            if response.status == 200:
                ready = True
                break
    except Exception:
        time.sleep(1)

if not ready:
    print(LOG_PATH.read_text(encoding='utf-8', errors='ignore')[-6000:])
    raise RuntimeError('Streamlit did not become healthy. Review the log above, then rerun Cells 1, 9, and 10.')

print('Streamlit is running and its health endpoint returned HTTP 200.')
if IN_COLAB:
    # Colab runs Python on a remote VM; localhost:8501 is NOT the browser URL.
    # The helpers create a session-specific Colab proxy link and an embedded preview.
    from google.colab import output
    output.serve_kernel_port_as_window(
        PORT, anchor_text='OPEN STREAMLIT DASHBOARD IN A NEW BROWSER TAB', skip_warning=True
    )
    output.serve_kernel_port_as_iframe(PORT, height='850')
    print('Use the clickable Colab proxy link above or the embedded dashboard preview. Do not open localhost:8501 on your own computer.')
    print('Keep this Colab runtime running while using the dashboard. The link is temporary, not a permanent public deployment.')
else:
    print(f'Open the dashboard in this same machine’s browser: http://localhost:{PORT}')


Streamlit is running and its health endpoint returned HTTP 200.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Use the clickable Colab proxy link above or the embedded dashboard preview. Do not open localhost:8501 on your own computer.
Keep this Colab runtime running while using the dashboard. The link is temporary, not a permanent public deployment.


In [11]:
# Cell 11 — Generate and download a ZIP of the outputs (tables, graphs, and dashboard source).
import zipfile

ZIP_PATH = WORK_DIR / 'retention_analytics_outputs.zip'
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    # Aggregate tables and interactive chart HTML files
    for path in OUTPUT_DIR.rglob('*'):
        if path.is_file() and path.suffix.lower() in {'.csv', '.html'}:
            zf.write(path, path.relative_to(OUTPUT_DIR.parent))

    # Include the exact Streamlit source used by the launch cell, but not the CSV.
    if APP_PATH.exists():
        zf.write(APP_PATH, 'retention_dashboard.py')

    zf.writestr(
        'README.txt',
        'Customer Engagement & Product Utilization Analytics outputs\n'
        'Includes aggregate CSV tables, interactive Plotly chart HTML files, and the Streamlit dashboard source.\n'
        'The source customer CSV and customer-level records are intentionally excluded.\n'
        'To reopen a chart, extract the ZIP and open an HTML file from retention_analytics_outputs/figures in a browser.\n'
    )

print(f'Created outputs ZIP: {ZIP_PATH}')
if IN_COLAB:
    from google.colab import files as colab_files
    colab_files.download(str(ZIP_PATH))
else:
    print(f'Download the ZIP from: {ZIP_PATH.resolve()}')

Created outputs ZIP: /content/retention_analytics_outputs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## If the dashboard does not open

Run the notebook cells in order. **Cell 10 checks that Streamlit is healthy** and then uses Google Colab's port-proxy helpers to show a clickable browser link and embedded preview. A Colab runtime runs remotely, so `http://localhost:8501` on your own computer will not connect to it.

The final cell creates a ZIP containing the aggregate CSV tables, interactive HTML graphs, and Streamlit dashboard source code. It does not generate a research paper or include the raw customer CSV. Keep the Colab runtime active while using the app; Colab proxy links are temporary.